## Feature Engineering

### Understanding Feature Extraction vs. Feature Engineering

- **Feature Extraction :** Refers to extracting raw physical measurements directly from raw data sources (e.g., locating facial landmarks and measuring raw pixel distances such as `eye_distance`, `mouth_width`, `face_width`, and `face_height`).
- **Feature Engineering :** Refers to constructing new, domain-informed derived variables from the raw measurements. Raw pixel distances vary depending on camera zoom, cropping, and resolution. Feature engineering addresses this scale dependency by deriving **proportional ratios**, which are **scale-invariant** and reflect underlying facial structure regardless of image resolution.

In [1]:
# Ensure working directory is set to project root
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Imports for data processing and numerical computations
import pandas as pd
import numpy as np

# Load dataset with raw facial measurements
measurements_csv_path = "data/processed/utkface_with_measurements.csv"
df = pd.read_csv(measurements_csv_path)

print("=== BEFORE DROPPING FAILED EXTRACTIONS ===")
print(f"Loaded dataset shape: {df.shape}")
measurement_cols = ['eye_distance', 'mouth_width', 'face_width', 'face_height']
print("\nNull counts for measurement columns:")
print(df[measurement_cols].isnull().sum())

=== BEFORE DROPPING FAILED EXTRACTIONS ===
Loaded dataset shape: (3000, 14)

Null counts for measurement columns:
eye_distance    2
mouth_width     2
face_width      2
face_height     2
dtype: int64


In [2]:
# Drop rows where any measurement column contains NaN (failed landmark extractions)
initial_row_count = len(df)
df_clean = df.dropna(subset=measurement_cols).reset_index(drop=True)
final_row_count = len(df_clean)
rows_dropped = initial_row_count - final_row_count

print("=== AFTER DROPPING FAILED EXTRACTIONS ===")
print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"Total rows dropped due to failed landmark detection: {rows_dropped}")

=== AFTER DROPPING FAILED EXTRACTIONS ===
Cleaned dataset shape: (2998, 14)
Total rows dropped due to failed landmark detection: 2


### Engineered Ratio Features Design

To ensure measurements are scale-invariant across varying image crops and resolutions, three key ratio features are derived:

1. **`face_aspect_ratio`**:
   - **Original Features Used:** `face_height`, `face_width`
   - **Formula:** `face_aspect_ratio = face_height / face_width`
   - **Engineering Reason:** Captures overall facial morphology (elongated vs. round face shape) independently of absolute image dimensions or bounding box scale.

2. **`eye_to_face_ratio`**:
   - **Original Features Used:** `eye_distance`, `face_width`
   - **Formula:** `eye_to_face_ratio = eye_distance / face_width`
   - **Engineering Reason:** Normalizes inter-ocular spacing relative to overall facial width, allowing direct comparison across subject faces regardless of distance from camera.

3. **`mouth_to_face_ratio`**:
   - **Original Features Used:** `mouth_width`, `face_width`
   - **Formula:** `mouth_to_face_ratio = mouth_width / face_width`
   - **Engineering Reason:** Normalizes mouth width relative to overall facial width to quantify facial component proportions in a scale-invariant manner.

In [3]:
# Compute scale-invariant ratio features using vectorized Pandas operations
df_clean['face_aspect_ratio'] = df_clean['face_height'] / df_clean['face_width']
df_clean['eye_to_face_ratio'] = df_clean['eye_distance'] / df_clean['face_width']
df_clean['mouth_to_face_ratio'] = df_clean['mouth_width'] / df_clean['face_width']

In [4]:
print("=== ENGINEERED FEATURE STATISTICS ===")
ratio_cols = ['face_aspect_ratio', 'eye_to_face_ratio', 'mouth_to_face_ratio']
print(df_clean[ratio_cols].describe())

print("\nDataFrame Head with Engineered Features:")
df_clean.head()

=== ENGINEERED FEATURE STATISTICS ===
       face_aspect_ratio  eye_to_face_ratio  mouth_to_face_ratio
count        2998.000000        2998.000000          2998.000000
mean            1.158987           0.464773             0.387999
std             0.080061           0.024431             0.037322
min             0.882454           0.363902             0.231000
25%             1.108917           0.447394             0.363160
50%             1.165218           0.463930             0.391641
75%             1.214007           0.480630             0.415128
max             1.568700           0.544142             0.488710

DataFrame Head with Engineered Features:


,image_name,age,filepath,gender_0,gender_1,race_0,race_1,race_2,race_3,race_4,eye_distance,mouth_width,face_width,face_height,face_aspect_ratio,eye_to_face_ratio,mouth_to_face_ratio
0,68_0_0_20170111205927835.jpg.chip.jpg,68,/Users/cipherxishant/Downloads/archive/UTKFace...,1,0,1,0,0,0,0,72.346710,63.034367,157.858292,200.633484,1.270972,0.458302,0.399310
1,69_1_0_20170110141308465.jpg.chip.jpg,69,/Users/cipherxishant/Downloads/archive/UTKFace...,0,1,1,0,0,0,0,84.516975,73.249702,170.779480,196.707672,1.151823,0.494890,0.428914
2,26_1_2_20170116184737269.jpg.chip.jpg,26,/Users/cipherxishant/Downloads/archive/UTKFace...,0,1,0,0,1,0,0,80.967552,67.399666,164.073837,187.231903,1.141144,0.493482,0.410789
3,34_0_3_20170119200815092.jpg.chip.jpg,34,/Users/cipherxishant/Downloads/archive/UTKFace...,1,0,0,0,0,1,0,82.894005,72.796394,171.991562,207.560822,1.206808,0.481966,0.423256
4,60_1_0_20170110141759687.jpg.chip.jpg,60,/Users/cipherxishant/Downloads/archive/UTKFace...,0,1,1,0,0,0,0,75.608727,62.865295,173.007172,209.123703,1.208757,0.437027,0.363368


In [6]:
# Export engineered DataFrame to data/processed/utkface_engineered.csv
output_engineered_path = "data/processed/utkface_engineered.csv"
df_clean.to_csv(output_engineered_path, index=False)

print(f"Engineered feature dataset saved successfully to '{output_engineered_path}' ({len(df_clean)} rows, {len(df_clean.columns)} columns).")

Engineered feature dataset saved successfully to 'data/processed/utkface_engineered.csv' (2998 rows, 17 columns).
